# DataHek OSS — 06 · Checkpoints & replay

Every run is stored with its plan and compiled SQL. Replay re-executes a
stored plan **deterministically — no LLM involved**.

In [1]:
import sys, os, tempfile
for _c in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.join(os.path.dirname(os.getcwd()), "notebooks")):
    if os.path.exists(os.path.join(_c, "datahek_demo.py")):
        sys.path.insert(0, _c)
        break
os.environ["DATAHEK_DB_PATH"] = os.path.join(tempfile.gettempdir(), "datahek_nb06.db")

from datahek_demo import build_demo_db, make_client, show_rows

db = build_demo_db()
client, container = make_client(db)

## Run two questions

In [2]:
for q in ["How many traces are there?", "What is the average duration per service?"]:
    client.post("/ask", json={"question": q, "connection_id": "conn_demo"})

for cp in client.get("/checkpoints").json():
    print(f"{cp['id'][-10:]}  rows={cp['row_count']:<3}  {cp['question']}")

f306a544e1  rows=4    What is the average duration per service?
cd85cf466b  rows=1    How many traces are there?
e647b6482d  rows=3    What is the average duration per service?
c042094034  rows=3    How many traces are there?


## Inspect a checkpoint's compiled SQL

In [3]:
cp_id = client.get("/checkpoints").json()[0]["id"]
detail = client.get(f"/checkpoints/{cp_id}").json()
print("question:", detail["question"])
print("sql:", detail["sql"])

question: What is the average duration per service?
sql: SELECT service, avg(duration_ms) AS avg_duration FROM traces GROUP BY service ORDER BY avg_duration DESC LIMIT 10


## Replay — same plan, same SQL, no model call

In [4]:
replay = client.post(f"/checkpoints/{cp_id}/replay").json()
print("replayed:", replay["replayed"])
print("sql:", replay["sql"])
show_rows(replay["rows"])

replayed: True
sql: SELECT service, avg(duration_ms) AS avg_duration FROM traces GROUP BY service ORDER BY avg_duration DESC LIMIT 10
service=auth-service | avg_duration=675.0
service=order-service | avg_duration=487.5
service=payment-api | avg_duration=176.0
service=inventory | avg_duration=57.5


## Audit trail
Every guardrail decision and execution is recorded as JSONL alongside the app.

In [5]:
import json, glob
audit_files = sorted(glob.glob(os.path.join(tempfile.gettempdir(), "datahek_nb06.db*")) + glob.glob("datahek-audit.jsonl"))
print("audit files:", audit_files or "(audit path defaults to ./datahek-audit.jsonl)")
try:
    with open("datahek-audit.jsonl", encoding="utf-8") as f:
        events = [json.loads(line) for line in f][-3:]
    for e in events:
        print(e["event_type"], "|", e.get("decision"))
except FileNotFoundError:
    print("(run from the repo root to see the audit file)")

audit files: ['C:\\Users\\azcom\\AppData\\Local\\Temp\\datahek_nb06.db', 'datahek-audit.jsonl']
query.execution | ALLOW
guardrail.decision | ALLOW
query.execution | ALLOW


**Takeaway:** runs are reproducible artifacts — inspect, diff, and replay
without re-planning or re-spending tokens.